# Part 1 – Text Pre-processing in NLP

Dataset: `trumptweets_small.csv`

This notebook covers:
- Removing URLs, emojis, and unwanted characters
- Tokenisation
- Stopword removal
- Lemmatisation & Stemming

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd
import numpy as np
import re

# NLTK setup
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import WordNetLemmatizer, PorterStemmer

stop_words = set(stopwords.words('english'))
stop_words.update(['@', 'RT'])   # custom additions

print('Libraries ready.')

Libraries ready.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Load Data

In [24]:
df = pd.read_csv('/content/drive/MyDrive/workshop8/trumptweets_small.csv')
df.head()

,id,link,content,date,retweets,favorites,mentions,hashtags,geo
0,1698308935,https://twitter.com/realDonaldTrump/status/169...,Be sure to tune in and watch Donald Trump on L...,2009-05-04 20:54:25,500,868,NaN,NaN,NaN
1,1701461182,https://twitter.com/realDonaldTrump/status/170...,Donald Trump will be appearing on The View tom...,2009-05-05 03:00:10,33,273,NaN,NaN,NaN
2,1737479987,https://twitter.com/realDonaldTrump/status/173...,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08 15:38:08,12,18,NaN,NaN,NaN
3,1741160716,https://twitter.com/realDonaldTrump/status/174...,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08 22:40:15,11,24,NaN,NaN,NaN
4,1773561338,https://twitter.com/realDonaldTrump/status/177...,"""My persona will never be that of a wallflower...",2009-05-12 16:07:28,1399,1965,NaN,NaN,NaN


In [25]:
df_text = df[['content']].dropna()
print('Rows:', len(df_text))
df_text.head()

Rows: 41122


,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post: Celebrity Apprentice Finale and...
4,"""My persona will never be that of a wallflower..."


## Helper Functions

In [26]:
# --- Remove URLs ---
def remove_urls(text):
    """Removes http/https and www links from text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub('', text)

# Quick test
test_url = 'Click here https://www.facebook.com/ for more info'
print(remove_urls(test_url))

Click here  for more info


In [27]:
# --- Remove Emojis ---
def remove_emoji(string):
    """Replaces emoji characters with a space."""
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', string)

test_emoji = 'Hello @siman 👋🏾, movie night? #MovieNight #🍱'
print(remove_emoji(test_emoji))

Hello @siman  , movie night? #MovieNight # 


In [28]:
# --- Remove All Unwanted Characters ---
def removeunwanted_characters(document):
    """Removes @mentions, #hashtags, punctuation, emojis, and extra spaces."""
    document = re.sub(r'@[A-Za-z0-9_]+', ' ', document)   # @mentions
    document = re.sub(r'#[A-Za-z0-9_]+', '', document)     # #hashtags
    document = re.sub(r'[^0-9A-Za-z ]', '', document)      # punctuation / symbols
    document = remove_emoji(document)
    document = document.replace('  ', ' ')
    return document.strip()

test_str = 'Hello @siman 👋🏾, still on for the movie??? #MovieNight #friday'
print(removeunwanted_characters(test_str))

Hello  still on for the movie


In [29]:
# Apply to full dataset
text_removed_unwanted = df_text['content'].apply(removeunwanted_characters)
text_removed_unwanted.head()

,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post Celebrity Apprentice Finale and ...
4,My persona will never be that of a wallflower ...


## Tokenisation

In [30]:
sentence = 'He did not try to navigate after the first bold flight, for the reaction had taken something out of his soul.'
tokens = word_tokenize(sentence)
print(tokens)

['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']


In [31]:
# --- Remove Punctuation (via RegexpTokenizer) ---
def remove_punct(text):
    """Tokenises text keeping only word characters (no punctuation)."""
    tokenizer = RegexpTokenizer(r'\w+')
    return tokenizer.tokenize(' '.join(text))

punc_test = 'He did not try: after the!!!! first bold flight,,,,,'
punc_tokens = word_tokenize(punc_test)
clean_tokens = remove_punct(punc_tokens)
print('Original :', punc_tokens)
print('Cleaned  :', clean_tokens)

Original : ['He', 'did', 'not', 'try', ':', 'after', 'the', '!', '!', '!', '!', 'first', 'bold', 'flight', ',', ',', ',', ',', ',']
Cleaned  : ['He', 'did', 'not', 'try', 'after', 'the', 'first', 'bold', 'flight']


## Stopword Removal

In [32]:
def remove_stopwords(text_tokens):
    """Filters stopwords from a list of tokens."""
    return [token for token in text_tokens if token not in stop_words]

test_inputs = ['He','did','not','try','to','navigate','after','the','first','bold','flight',',','for','the','reaction','had','taken','something','out','of','his','soul','.']
print('Original :', test_inputs)
print('Filtered :', remove_stopwords(test_inputs))

Original : ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']
Filtered : ['He', 'try', 'navigate', 'first', 'bold', 'flight', ',', 'reaction', 'taken', 'something', 'soul', '.']


## Text Normalisation: Lemmatisation & Stemming

In [33]:
# --- Lemmatisation ---
def lemmatization(token_text):
    """Reduces each token to its base verb form using WordNetLemmatizer."""
    wordnet = WordNetLemmatizer()
    return [wordnet.lemmatize(token, pos='v') for token in token_text]

print(lemmatization('Should we go walking or swimming'.split()))

['Should', 'we', 'go', 'walk', 'or', 'swim']


In [34]:
# --- Stemming ---
def stemming(text):
    """Reduces each token to its root form using PorterStemmer."""
    porter = PorterStemmer()
    return [porter.stem(word) for word in text]

words = ['Connects','Connecting','Connections','Connected','Connection','Connect']
print('Lemmatised:', lemmatization(words))
print('Stemmed   :', stemming(words))

Lemmatised: ['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connect']
Stemmed   : ['connect', 'connect', 'connect', 'connect', 'connect', 'connect']


In [35]:
# --- Lowercase ---
def lower_order(text):
    """Converts all text to lowercase."""
    return text.lower()

print(lower_order('This Is some Normalized TEXT'))

this is some normalized text


## Exercise 1 – Text Cleaning Pipeline

Combines every step into one function applied to the sentiment dataset.

In [36]:
data = pd.read_csv('/content/drive/MyDrive/workshop8/trum_tweet_sentiment_analysis.csv', encoding='ISO-8859-1')
data.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [37]:
def text_cleaning_pipeline(dataset, rule='lemmatize'):
    """Full cleaning pipeline: lowercase → remove URLs/emojis/noise → tokenise → remove stopwords → lemmatise or stem."""
    data = lower_order(dataset)
    data = remove_urls(data)
    data = remove_emoji(data)
    data = removeunwanted_characters(data)
    tokens = data.split()
    tokens = [word for word in tokens if word not in stop_words]
    if rule == 'lemmatize':
        tokens = lemmatization(tokens)
    elif rule == 'stem':
        tokens = stemming(tokens)
    else:
        print("Pick between 'lemmatize' or 'stem'")
    return ' '.join(tokens)

# Quick test
sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? #sushiBros"
print(text_cleaning_pipeline(sample))

hello still want us hit new sushi spot


In [38]:
# Test on a single tweet from the dataset
test = data['text'][0]
print('Original :', test)
print('Cleaned  :', text_cleaning_pipeline(test))

Original : RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrumpÂ https://t.co/gFBvUkMX9z
Cleaned  : rt drain swamp taxpayer dollars trip advertise properties


In [39]:
# Apply to all tweets (may take a minute)
cleaned_tokens = data['text'].apply(lambda tweet: text_cleaning_pipeline(tweet))
print('Done. Sample output:')
print(cleaned_tokens[0])

Done. Sample output:
rt drain swamp taxpayer dollars trip advertise properties
